# Exercise 1: Identifying Limitations of Traditional Language Models

* **The Distance Problem:** Traditional models (like RNNs) process text word-by-word sequentially. If there are many words between "scientist" and "discovery," the model tends to "forget" the initial connection.
* **Impact:** This affects tasks like question answering or summarization because the model loses the thread of the sentence.
* **The Solution (Attention):** The attention mechanism allows any word to connect directly to any other word, regardless of the distance between them.

# Exercise 2: Exploring the Impact of Attention in Transformers

* **Chosen Task:** Question Answering.
* **How it Works:** Models like BERT and GPT use **self-attention** to evaluate which words are most important relative to each other simultaneously.
* **Practical Example:** Helps resolve words with multiple meanings (like a river "bank" vs. a financial "bank") based on immediate context.
* **Comparison:** Models with attention vastly outperform older models by eliminating the sequential bottleneck.

In [ ]:
import faiss
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer, pipeline

# 1. Knowledge source (sample documents)
documents = [
    "Retrieval-Augmented Generation (RAG) combines external document retrieval with text generation.",
    "BERT is a transformer-based bidirectional model pretrained on a large text corpus.",
    "Vector databases like FAISS enable lightning-fast similarity search over dense vector embeddings."
]

# 2. Generate embeddings using pre-trained BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

doc_embeddings = np.array([get_bert_embedding(doc) for doc in documents])

# 3. Store embeddings in FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# 4. Retriever implementation
def retrieve_documents(query, k=1):
    query_emb = np.array([get_bert_embedding(query)])
    distances, indices = index.search(query_emb, k)
    return [documents[i] for i in indices[0]]

# 5. Generator initialization
generator = pipeline("text2text-generation", model="google/flan-t5-small")

# 6. RAG Pipeline execution
def answer_question(query):
    retrieved_docs = retrieve_documents(query, k=2)
    context = " ".join(retrieved_docs)
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer based on context:"
    response = generator(prompt, max_length=128)[0]["generated_text"]
    return response

# Test the RAG system
query_text = "What does RAG combine?"
print(f"Query: {query_text}")
print(f"Answer: {answer_question(query_text)}")

# Exercise 5: Comparing RAG with Traditional Generative Models

| Feature | Traditional Generative Models (GPT) | RAG (Retrieval-Augmented Generation) |
| :--- | :--- | :--- |
| **Where it stores information** | In its own weights (internal memory) | In an external database (vector database) |
| **Factual accuracy** | Prone to inventing (hallucinating) on new topics | Highly accurate because it relies on real documents |
| **Updating knowledge** | Requires retraining the model (very costly) | Updated instantly by adding or removing documents |

# Exercise 6: Exploring the Future of RAG in NLP

* **Current Trends:** Multimodal RAG (combining text, images, and audio), and autonomous agent systems that search and verify information multiple times before answering.
* **Challenges:** Improving speed, handling millions of documents without losing accuracy, and reducing search bias.